0️⃣ Imports & setup

In [9]:
import jax
import jax.numpy as jnp
from jax import random
import numpy as np

import numpyro
import numpyro.distributions as dist
from numpyro.infer import SVI, Trace_ELBO, Predictive
from numpyro.infer.autoguide import AutoNormal
from numpyro.optim import Adam


1️⃣ OU scan (pure JAX, no randomness inside)

In [10]:
def ou_scan(theta, sigma_f, f0, dts, eps):
    """
    theta: (k,)
    sigma_f: (k,)
    f0: (k,)
    dts: (T-1,)
    eps: (T-1, k)
    """

    def step(f_prev, inputs):
        dt, eps_t = inputs
        a = jnp.exp(-theta * dt)
        q = sigma_f * jnp.sqrt((1.0 - a**2) / (2.0 * theta))
        f_new = a * f_prev + q * eps_t
        return f_new, f_new

    _, f_seq = jax.lax.scan(step, f0, (dts, eps))
    return jnp.concatenate([f0[None, :], f_seq], axis=0)  # (T, k)


2️⃣ NumPyro OU-DFA model (scalable & SVI-safe)

In [11]:
def ou_dfa_model(Y_proj, time_deltas, k_factors):
    """
    Y_proj: (N, T, r)  projected observations
    time_deltas: (N, T-1)
    """

    N, T, r = Y_proj.shape
    k = k_factors

    # -------------------------
    # Factor loadings (r x k)
    # -------------------------
    B = numpyro.sample(
        "B",
        dist.Normal(0, 0.3).expand((r, k)).to_event(2)
    )

    # -------------------------
    # OU parameters (shared)
    # -------------------------
    theta = numpyro.sample(
        "theta",
        dist.Exponential(1.0).expand((k,)).to_event(1)
    )

    sigma_f = numpyro.sample(
        "sigma_f",
        dist.HalfNormal(1.0).expand((k,)).to_event(1)
    )

    # -------------------------
    # Observation noise
    # -------------------------
    sigma_y = numpyro.sample(
        "sigma_y",
        dist.HalfNormal(1.0).expand((r,)).to_event(1)
    )

    # -------------------------
    # Initial latent states
    # -------------------------
    f0 = numpyro.sample(
        "f0",
        dist.Normal(0, 1).expand((N, k)).to_event(2)
    )

    # -------------------------
    # OU innovation noise
    # -------------------------
    eps = numpyro.sample(
        "eps",
        dist.Normal(0, 1).expand((N, T-1, k)).to_event(3)
    )

    # -------------------------
    # OU dynamics (vmap over subjects)
    # -------------------------
    F = jax.vmap(ou_scan, in_axes=(None, None, 0, 0, 0))(
        theta,
        sigma_f,
        f0,
        time_deltas,
        eps
    )  # (N, T, k)

    # -------------------------
    # Observation model (low-rank)
    # -------------------------
    mu = jnp.einsum("ntk,rk->ntr", F, B)

    numpyro.sample(
        "y",
        dist.Normal(mu, sigma_y).to_event(2),
        obs=Y_proj
    )


3️⃣ Synthetic data generator (large-p safe)

In [12]:
def simulate_ou_dfa(
    key,
    N=200,
    T=4,
    p=10000,
    r=100,
    k=20,
):
    key, k1, k2, k3, k4 = random.split(key, 5)

    # Fixed random projection (p -> r)
    W = random.normal(k1, (p, r)) / jnp.sqrt(p)

    B_true = random.normal(k2, (r, k)) * 0.5
    theta_true = random.exponential(k3, (k,))
    sigma_f_true = jnp.ones(k) * 0.5
    sigma_y_true = jnp.ones(r) * 0.3

    # Irregular ages
    times = jnp.sort(
        random.uniform(k4, (N, T)) * 50 + 30,
        axis=1
    )
    dts = jnp.diff(times, axis=1)

    # Simulate latent factors
    def sim_subject(key, dts):
        f = []
        f0 = random.normal(key, (k,))
        f.append(f0)
        for dt in dts:
            a = jnp.exp(-theta_true * dt)
            q = sigma_f_true * jnp.sqrt((1 - a**2) / (2 * theta_true))
            f.append(a * f[-1] + q * random.normal(key, (k,)))
        return jnp.stack(f)

    keys = random.split(key, N)
    F = jax.vmap(sim_subject)(keys, dts)  # (N, T, k)

    # Observations in projected space
    Y_proj = jnp.einsum("ntk,rk->ntr", F, B_true)
    Y_proj += sigma_y_true * random.normal(key, Y_proj.shape)

    return Y_proj, dts, B_true, F


In [14]:
def make_ou_dfa_model(k_factors):
    def model(Y_proj, time_deltas):
        N, T, r = Y_proj.shape
        k = k_factors

        B = numpyro.sample(
            "B",
            dist.Normal(0, 0.3).expand((r, k)).to_event(2)
        )

        theta = numpyro.sample(
            "theta",
            dist.Exponential(1.0).expand((k,)).to_event(1)
        )

        sigma_f = numpyro.sample(
            "sigma_f",
            dist.HalfNormal(1.0).expand((k,)).to_event(1)
        )

        sigma_y = numpyro.sample(
            "sigma_y",
            dist.HalfNormal(1.0).expand((r,)).to_event(1)
        )

        f0 = numpyro.sample(
            "f0",
            dist.Normal(0, 1).expand((N, k)).to_event(2)
        )

        eps = numpyro.sample(
            "eps",
            dist.Normal(0, 1).expand((N, T-1, k)).to_event(3)
        )

        F = jax.vmap(ou_scan, in_axes=(None, None, 0, 0, 0))(
            theta, sigma_f, f0, time_deltas, eps
        )

        mu = jnp.einsum("ntk,rk->ntr", F, B)

        numpyro.sample(
            "y",
            dist.Normal(mu, sigma_y).to_event(2),
            obs=Y_proj
        )

    return model


4️⃣ Training with SVI (correct RNG usage)

In [16]:
# Simulate data
key = random.PRNGKey(0)
Y_proj, dts, B_true, F_true = simulate_ou_dfa(
    key,
    N=200,
    T=4,
    p=10000,
    r=100,
    k=20,
)

# Guide & optimizer
guide = AutoNormal(ou_dfa_model)
optimizer = Adam(1e-3)

svi = SVI(
    ou_dfa_model,
    guide,
    optimizer,
    loss=Trace_ELBO()
)

# Initialize
key, init_key = random.split(key)
state = svi.init(
    init_key,
    Y_proj=Y_proj,
    time_deltas=dts,
    k_factors=20
)

# # Train
# for step in range(2000):
#     key, step_key = random.split(key)
#     state, loss = svi.update(
#         state,
#         step_key,
#         Y_proj=Y_proj,
#         time_deltas=dts,
#         k_factors=20
#     )
#     if step % 200 == 0:
#         print(f"step {step}, loss {loss:.2f}")

# params = svi.get_params(state)


In [17]:
model = make_ou_dfa_model(k_factors=20)
guide = AutoNormal(model)

svi = SVI(
    model,
    guide,
    Adam(1e-3),
    Trace_ELBO()
)

key, init_key = random.split(key)
state = svi.init(
    init_key,
    Y_proj,
    dts
)

for step in range(2000):
    key, step_key = random.split(key)
    state, loss = svi.update(
        state,
        Y_proj,
        dts
    )
    if step % 200 == 0:
        print(f"step {step}, loss {loss:.2f}")


step 0, loss 47513592.00
step 200, loss 14872943.00
step 400, loss 9027361.00


E0128 10:32:32.007317 1645890 execution_engine.cc:54] LLVM compilation error: Cannot allocate memory
E0128 10:32:32.007413 1645886 execution_engine.cc:54] LLVM compilation error: Cannot allocate memory
E0128 10:32:32.007639 1645883 execution_engine.cc:54] LLVM compilation error: Cannot allocate memory
E0128 10:32:32.010694 1645889 execution_engine.cc:54] LLVM compilation error: Cannot allocate memory
E0128 10:32:32.020091 1645881 execution_engine.cc:54] LLVM compilation error: Cannot allocate memory
E0128 10:32:32.026621 1645880 execution_engine.cc:54] LLVM compilation error: Cannot allocate memory
E0128 10:32:32.039337 1645876 execution_engine.cc:54] LLVM compilation error: Cannot allocate memory
E0128 10:32:32.040131 1645887 execution_engine.cc:54] LLVM compilation error: Cannot allocate memory


XlaRuntimeError: INTERNAL: Failed to materialize symbols: { (<xla_jit_dylib_8>, { bitcast_dynamic-update-slice_fusion.4 }) }

5️⃣ Factor loading recovery (up to rotation)

In [ ]:
from scipy.linalg import orthogonal_procrustes

params = svi.get_params(state)
B_est = np.array(params["B"])
B_true_np = np.array(B_true)

R, _ = orthogonal_procrustes(B_est, B_true_np)
B_aligned = B_est @ R

corr = np.corrcoef(
    B_aligned.ravel(),
    B_true_np.ravel()
)[0, 1]

print("Loading recovery correlation:", corr)


6️⃣ Posterior predictive check

In [ ]:
predictive = Predictive(
    ou_dfa_model,
    guide=guide,
    params=params,
    num_samples=100
)

ppc = predictive(
    random.PRNGKey(1),
    Y_proj=None,
    time_deltas=dts,
    k_factors=20
)

Y_rep = ppc["y"]  # (S, N, T, r)

print("PPC mean var:", Y_rep.var(axis=(0,1,2)).mean())
print("Observed var:", Y_proj.var())


In [18]:
import jax
import jax.numpy as jnp
from jax import random
import numpy as np
import numpyro
from numpyro import handlers
import numpyro.distributions as dist
from numpyro.infer import SVI, Trace_ELBO, AutoDiagonalNormal
from numpyro.optim import Adam

# ----------------------------
# Simulate OU-DFA data
# ----------------------------
def simulate_ou_dfa(key, N=100, T=4, p=1000, r=50, k=10):
    """Simulate OU dynamic factor analysis data."""
    key, subkey = random.split(key)
    B_true = random.normal(subkey, (p, r)) * 0.5

    key, subkey = random.split(key)
    F_true = random.normal(subkey, (N, T, r))  # latent factors

    # OU dynamics along time
    theta_true = 0.3
    for i in range(N):
        for t in range(1, T):
            dt = 1.0  # assume unit delta
            F_true = F_true.at[i, t].set(F_true[i, t-1]*jnp.exp(-theta_true*dt) + 
                                         random.normal(key, (r,))*jnp.sqrt(1 - jnp.exp(-2*theta_true*dt)))
    
    # Project to high-dim observations
    key, subkey = random.split(key)
    Y = jnp.einsum('ntr,pr->ntp', F_true, B_true)
    Y += 0.05 * random.normal(subkey, Y.shape)  # add noise
    dts = jnp.ones((T-1,))  # simple uniform time delta
    return Y, dts, B_true, F_true

# ----------------------------
# OU-DFA Model
# ----------------------------
def ou_dfa_model(Y, dts, k_factors, r_proj=50, subsample_size=None):
    """
    Y: (N, T, p)
    dts: (T-1,)
    k_factors: number of latent factors
    r_proj: projection rank for observations
    subsample_size: minibatch size for subjects
    """
    N, T, p = Y.shape
    key = numpyro.sample("key", dist.Bernoulli(0.5))  # dummy to use rng

    # -----------------------------
    # Low-rank projection
    # -----------------------------
    B = numpyro.sample("B", dist.Normal(0, 1).expand((p, r_proj)).to_event(2))
    sigma_y = numpyro.sample("sigma_y", dist.HalfNormal(0.1))
    
    # -----------------------------
    # OU latent dynamics
    # -----------------------------
    theta = numpyro.sample("theta", dist.Exponential(1.0))
    sigma_f = numpyro.sample("sigma_f", dist.HalfNormal(1.0))

    def ou_step(f_prev, dt):
        a = jnp.exp(-theta * dt)
        q = sigma_f * jnp.sqrt(1 - a**2)
        eps = numpyro.sample("eps", dist.Normal(0, 1).expand((k_factors,)).to_event(1))
        return a * f_prev + q * eps, None

    # -----------------------------
    # Subject-level dynamics
    # -----------------------------
    if subsample_size is None:
        subj_idx = jnp.arange(N)
    else:
        subj_idx = numpyro.subsample(jnp.arange(N), event_size=subsample_size)

    for i in subj_idx:
        f0 = numpyro.sample(f"f0_{i}", dist.Normal(0, 1).expand((k_factors,)).to_event(1))
        f_seq, _ = jax.lax.scan(ou_step, f0, dts)
        f_full = jnp.concatenate([f0[None, :], f_seq], axis=0)  # (T, k)
        mu = jnp.dot(f_full, B.T)  # (T, p)
        numpyro.sample(f"y_{i}", dist.Normal(mu, sigma_y).to_event(2), obs=Y[i])

# ----------------------------
# Main training routine
# ----------------------------
def main():
    key = random.PRNGKey(0)
    N, T, p, r, k = 200, 4, 1000, 50, 20
    Y, dts, B_true, F_true = simulate_ou_dfa(key, N, T, p, r, k)

    # AutoDiagonal guide + minibatch for N
    guide = AutoDiagonalNormal(lambda Y, dts: ou_dfa_model(Y, dts, k_factors=k, r_proj=r, subsample_size=32))

    optimizer = Adam(1e-3)
    svi = SVI(lambda Y, dts: ou_dfa_model(Y, dts, k_factors=k, r_proj=r, subsample_size=32),
              guide,
              optimizer,
              loss=Trace_ELBO(num_particles=1))

    state = svi.init(key, Y, dts)

    @jax.jit
    def svi_step(state, key):
        return svi.update(state, Y, dts)

    for step in range(2000):
        key, step_key = random.split(key)
        state, loss = svi_step(state, step_key)
        if step % 200 == 0:
            print(f"step {step}, loss {loss:.2f}")

    params = svi.get_params(state)
    print("Training done.")

if __name__ == "__main__":
    main()


ImportError: cannot import name 'AutoDiagonalNormal' from 'numpyro.infer' (/u/zwu1/.conda/envs/gnpc/lib/python3.12/site-packages/numpyro/infer/__init__.py)

In [20]:
import jax
import jax.numpy as jnp
from jax import random
import numpy as np
import matplotlib.pyplot as plt

import numpyro
import numpyro.distributions as dist
from numpyro.infer import SVI, Trace_ELBO
from numpyro.infer.autoguide import AutoDiagonalNormal
from numpyro.optim import Adam
from numpyro import handlers

# ----------------------------
# Simulate OU-DFA data
# ----------------------------
def simulate_ou_dfa(key, N=100, T=4, p=1000, r=50, k=10):
    key, subkey = random.split(key)
    B_true = random.normal(subkey, (p, r)) * 0.5
    key, subkey = random.split(key)
    F_true = random.normal(subkey, (N, T, r))  # latent factors

    theta_true = 0.3
    dt = 1.0
    for i in range(N):
        for t in range(1, T):
            key, subkey = random.split(key)
            F_true = F_true.at[i, t].set(
                F_true[i, t-1] * jnp.exp(-theta_true * dt)
                + random.normal(subkey, (r,)) * jnp.sqrt(1 - jnp.exp(-2*theta_true*dt))
            )
    key, subkey = random.split(key)
    Y = jnp.einsum('ntr,pr->ntp', F_true, B_true)
    Y += 0.05 * random.normal(subkey, Y.shape)  # observation noise
    dts = jnp.ones((T-1,))
    return Y, dts, B_true, F_true

# ----------------------------
# OU-DFA Model
# ----------------------------
def ou_dfa_model(Y, dts, k_factors, r_proj=50, subsample_size=None):
    N, T, p = Y.shape

    B = numpyro.sample("B", dist.Normal(0, 1).expand((p, r_proj)).to_event(2))
    sigma_y = numpyro.sample("sigma_y", dist.HalfNormal(0.1))
    theta = numpyro.sample("theta", dist.Exponential(1.0))
    sigma_f = numpyro.sample("sigma_f", dist.HalfNormal(1.0))

    if subsample_size is None:
        subj_idx = jnp.arange(N)
    else:
        subj_idx = numpyro.subsample(jnp.arange(N), event_size=subsample_size)

    def ou_step(f_prev, dt):
        a = jnp.exp(-theta * dt)
        q = sigma_f * jnp.sqrt(1 - a**2)
        eps = numpyro.sample("eps", dist.Normal(0, 1).expand((k_factors,)).to_event(1))
        return a * f_prev + q * eps, None

    for i in subj_idx:
        f0 = numpyro.sample(f"f0_{i}", dist.Normal(0, 1).expand((k_factors,)).to_event(1))
        f_seq, _ = jax.lax.scan(ou_step, f0, dts)
        f_full = jnp.concatenate([f0[None, :], f_seq], axis=0)
        mu = jnp.dot(f_full, B.T)
        numpyro.sample(f"y_{i}", dist.Normal(mu, sigma_y).to_event(2), obs=Y[i])

# ----------------------------
# Posterior predictive function
# ----------------------------
def posterior_predictive(params, key, Y_shape, dts, k_factors, r_proj=50):
    N, T, p = Y_shape
    B_est = params['AutoDiagonalNormal']['B_loc']
    sigma_y_est = params['AutoDiagonalNormal']['sigma_y_loc']
    theta_est = params['AutoDiagonalNormal']['theta_loc']
    sigma_f_est = params['AutoDiagonalNormal']['sigma_f_loc']

    Y_pred = []
    for i in range(N):
        f = [jnp.zeros(k_factors)]
        for dt in dts:
            a = jnp.exp(-theta_est * dt)
            q = sigma_f_est * jnp.sqrt(1 - a**2)
            key, subkey = random.split(key)
            f_new = a * f[-1] + q * random.normal(subkey, (k_factors,))
            f.append(f_new)
        f_full = jnp.stack(f, axis=0)
        key, subkey = random.split(key)
        Y_i = jnp.dot(f_full, B_est.T) + sigma_y_est * random.normal(subkey, (T, p))
        Y_pred.append(Y_i)
    return jnp.stack(Y_pred, axis=0)

# ----------------------------
# Factor recovery
# ----------------------------
def factor_recovery_plot(B_true, B_est):
    from scipy.linalg import orthogonal_procrustes
    R, _ = orthogonal_procrustes(B_est, B_true)
    B_aligned = B_est @ R
    plt.figure(figsize=(6,6))
    plt.scatter(B_true.flatten(), B_aligned.flatten(), alpha=0.5)
    plt.xlabel("True loadings")
    plt.ylabel("Recovered loadings (aligned)")
    plt.plot([-1,1],[-1,1],'r--')
    plt.title("Factor Recovery Check")
    plt.show()

# ----------------------------
# Quantitative RMSE
# ----------------------------
def compute_rmse(Y_true, Y_pred):
    rmse = jnp.sqrt(jnp.mean((Y_true - Y_pred)**2))
    print(f"Posterior predictive RMSE: {rmse:.4f}")
    return rmse

# ----------------------------
# Main training + validation
# ----------------------------
def main():
    key = random.PRNGKey(0)
    N, T, p, r, k = 100, 4, 1000, 50, 20
    Y, dts, B_true, F_true = simulate_ou_dfa(key, N, T, p, r, k)

    # AutoDiagonalNormal guide with minibatch
    guide = AutoDiagonalNormal(lambda Y, dts: ou_dfa_model(Y, dts, k_factors=k, r_proj=r, subsample_size=32))

    optimizer = Adam(1e-3)
    svi = SVI(lambda Y, dts: ou_dfa_model(Y, dts, k_factors=k, r_proj=r, subsample_size=32),
              guide,
              optimizer,
              loss=Trace_ELBO(num_particles=1))

    state = svi.init(key, Y, dts)

    @jax.jit
    def svi_step(state):
        return svi.update(state, Y, dts)

    for step in range(2000):
        state, loss = svi_step(state)
        if step % 200 == 0:
            print(f"step {step}, loss {loss:.2f}")

    params = svi.get_params(state)
    B_est = params['AutoDiagonalNormal']['B_loc']

    # Factor recovery plot
    factor_recovery_plot(B_true, B_est)

    # Posterior predictive check
    key, subkey = random.split(key)
    Y_pred = posterior_predictive(params, subkey, Y.shape, dts, k, r_proj=r)

    # Quantitative RMSE
    compute_rmse(Y, Y_pred)

    print("Training, factor recovery, and posterior predictive validation completed.")

if __name__ == "__main__":
    main()


ImportError: libjpeg.so.9: failed to map segment from shared object